# Merge City Listings for ML
**Inputs:** `../Data/interim/{city}_listings_clean.parquet` (Madrid · Barcelona · Málaga)  
**Output:** `../Data/processed/listings_all_cities.parquet`  
**Prerequisite:** run each city's cleaning notebook first.

Concatenates the three cleaned listing files, adds a `city` column as the first column,
runs basic validation, and saves a single parquet ready for ML feature engineering.

In [ ]:
import pathlib
import numpy as np
import pandas as pd

CITY_ORDER = ['Madrid', 'Barcelona', 'Málaga']

_PATHS = {
    'Madrid':    '../Data/interim/madrid_listings_clean.parquet',
    'Barcelona': '../Data/interim/barcelona_listings_clean.parquet',
    'Málaga':    '../Data/interim/malaga_listings_clean.parquet',
}

OUTPUT_PATH = pathlib.Path('../Data/processed/listings_all_cities.parquet')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

## 1. Load and tag each city

In [ ]:
frames = []
for city in CITY_ORDER:
    path = pathlib.Path(_PATHS[city])
    if not path.exists():
        print(f'[WARNING] {city}: file not found — {path}')
        continue
    df = pd.read_parquet(path)
    if df.empty:
        print(f'[WARNING] {city}: parquet is empty')
        continue
    df.insert(0, 'city', city)
    frames.append(df)
    print(f'{city:12s}  {len(df):>6,} listings | {df.shape[1]} cols')

if not frames:
    raise RuntimeError('No city data loaded — check parquet paths above.')

## 2. Concatenate

In [ ]:
all_df = pd.concat(frames, join='outer', ignore_index=True)
all_df['city'] = pd.Categorical(all_df['city'], categories=CITY_ORDER, ordered=True)

print(f'Combined: {all_df.shape[0]:,} rows × {all_df.shape[1]} cols')
all_df['city'].value_counts().reindex(CITY_ORDER)

## 3. Sanity checks

In [ ]:
# Duplicate listing IDs across cities are expected (same id space per InsideAirbnb source)
# but within a city they should be unique
dup_within_city = (
    all_df.groupby('city')['id']
    .apply(lambda s: s.duplicated().sum())
)
print('Duplicate IDs within city:')
print(dup_within_city.to_string())

In [ ]:
# Missing value summary — columns with >5% nulls
null_pct = all_df.isnull().mean().mul(100).round(2)
high_null = null_pct[null_pct > 5].sort_values(ascending=False)
print(f'Columns with >5% nulls ({len(high_null)}/{all_df.shape[1]}):')
print(high_null.to_string() if not high_null.empty else 'None')

In [ ]:
# Column dtype overview
dtype_summary = all_df.dtypes.astype(str).value_counts().rename('count')
print('Dtype distribution:')
print(dtype_summary.to_string())

In [ ]:
# Key ML target and feature range check
for col in ['price', 'estimated_revenue_l365d', 'estimated_occupancy_l365d', 'review_scores_rating']:
    if col not in all_df.columns:
        continue
    print(f'\n{col}:')
    print(all_df.groupby('city')[col].describe().round(2).to_string())

## 4. Save

In [ ]:
all_df.to_parquet(OUTPUT_PATH, index=False)
size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f'Saved → {OUTPUT_PATH}  ({size_mb:.1f} MB)')
print(f'Shape : {all_df.shape[0]:,} rows × {all_df.shape[1]} cols')
print(f'Cols  : city is first — {list(all_df.columns[:6])} ...')